# Uji Kelayakan Validasi Silang Antar Dataset (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Judul:** Prediksi Risiko Stroke Menggunakan Logistic Regression dan Gradient Boosting dengan Interpretasi Explainable AI

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Dataset utama kami mencantumkan asal-usulnya sebagai "(Confidential Source)": tidak
diketahui berasal dari rumah sakit, negara, atau tahun berapa. Untuk menutupi kelemahan
itu, kami menambahkan dataset kedua dengan asal-usul yang jelas.

**Kedua dataset TIDAK digabungkan barisnya.** Menggabungkan dua dataset berarti
menyatukan populasi berbeda, definisi klinis berbeda, dan cara pengumpulan berbeda ke
dalam satu tabel, yang dihasilkan bukan data gabungan, melainkan data karangan.

Dataset kedua dipakai sebagai **penguji independen**: model dilatih pada satu sumber,
lalu diuji pada sumber yang sama sekali berbeda. Ini disebut **validasi eksternal**,
dan merupakan pengujian yang jauh lebih ketat daripada data uji biasa.


## 1. Dua Sumber Data

| | Kaggle (fedesoriano) | CDC BRFSS 2015 |
|---|---|---|
| Baris | 5.110 | 253.680 |
| Kasus stroke | 249 | 10.292 |
| Asal-usul | "(Confidential Source)" | survei resmi CDC, terdokumentasi |
| Missing value | ada | tidak ada |

Dataset CDC aslinya disusun untuk prediksi diabetes, tetapi memuat kolom `Stroke` yang
kami jadikan target.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, recall_score, precision_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

CDC = "https://archive.ics.uci.edu/static/public/891/data.csv"
KAGGLE = ("https://raw.githubusercontent.com/ray-project/raydp/master/"
          "tutorials/dataset/healthcare-dataset-stroke-data.csv")

FITUR = ["sex", "umur_kel", "tekanan_darah_tinggi", "penyakit_jantung", "bmi", "perokok"]

# CDC mengelompokkan umur ke 13 kelompok; ini batas atas tiap kelompok (18-24, 25-29, ...)
BATAS_UMUR = [24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79]

## 2. Penyelarasan Fitur

Enam fitur tersedia di kedua dataset. Penamaannya berbeda, jadi harus diselaraskan
lebih dulu.

| Fitur selaras | Kaggle | CDC BRFSS |
|---|---|---|
| Jenis kelamin | `gender` | `Sex` |
| Kelompok usia | `age` → dikelompokkan ke skala 1–13 | `Age` |
| Hipertensi | `hypertension` | `HighBP` |
| Penyakit jantung | `heart_disease` | `HeartDiseaseorAttack` |
| BMI | `bmi` | `BMI` |
| Perokok | `smoking_status` | `Smoker` |

Baris berusia di bawah 18 tahun dibuang karena survei CDC hanya mencakup orang dewasa,
begitu pula baris dengan `smoking_status` = "Unknown" karena tidak bisa dipetakan.

In [2]:
def dari_cdc():
    d = pd.read_csv(CDC)
    return pd.DataFrame({
        "sex": d.Sex,
        "umur_kel": d.Age,
        "tekanan_darah_tinggi": d.HighBP,
        "penyakit_jantung": d.HeartDiseaseorAttack,
        "bmi": d.BMI,
        "perokok": d.Smoker,
        "stroke": d.Stroke,
    })


def dari_kaggle():
    d = pd.read_csv(KAGGLE)
    d = d[(d.age >= 18) & (d.gender != "Other") & (d.smoking_status != "Unknown")].copy()
    d["bmi"] = d.bmi.fillna(d.bmi.median())
    return pd.DataFrame({
        "sex": (d.gender == "Male").astype(int),
        "umur_kel": np.digitize(d.age, BATAS_UMUR) + 1,   # samakan ke skala 1-13 milik CDC
        "tekanan_darah_tinggi": d.hypertension,
        "penyakit_jantung": d.heart_disease,
        "bmi": d.bmi,
        "perokok": d.smoking_status.isin(["formerly smoked", "smokes"]).astype(int),
        "stroke": d.stroke,
    })

In [3]:
cdc = dari_cdc()
kaggle = dari_kaggle()

print(f"CDC BRFSS : {len(cdc):,} baris, positif {cdc.stroke.sum():,} ({cdc.stroke.mean()*100:.2f}%)")
print(f"Kaggle    : {len(kaggle):,} baris, positif {kaggle.stroke.sum():,} ({kaggle.stroke.mean()*100:.2f}%)")
print()
print("Setelah penyelarasan, dataset Kaggle menyusut karena baris anak-anak dan")
print("baris tanpa keterangan merokok dibuang.")
kaggle.head()

CDC BRFSS : 253,680 baris, positif 10,292 (4.06%)
Kaggle    : 3,391 baris, positif 202 (5.96%)

Setelah penyelarasan, dataset Kaggle menyusut karena baris anak-anak dan
baris tanpa keterangan merokok dibuang.


,sex,umur_kel,tekanan_darah_tinggi,penyakit_jantung,bmi,perokok,stroke
0,1,10,0,1,36.6,1,1
1,0,9,0,0,29.3,0,1
2,1,13,0,1,32.5,0,1
3,0,7,0,0,34.4,1,1
4,0,13,1,0,24.0,0,1


## 3. Latih di CDC, Uji di Kaggle

**Ambang keputusan ditetapkan dari data latih**, tidak pernah dari data uji. Kalau
ditetapkan dari data uji, itu kebocoran informasi dan hasilnya tidak sah.

In [4]:
MODEL = {
    "Logistic Regression": lambda: make_pipeline(StandardScaler(),
                                                 LogisticRegression(max_iter=2000, random_state=42)),
    "Gradient Boosting":   lambda: make_pipeline(StandardScaler(),
                                                 GradientBoostingClassifier(random_state=42)),
}

hasil = []
model_terlatih = {}

for nama, buat in MODEL.items():
    m = buat().fit(cdc[FITUR], cdc.stroke)
    model_terlatih[nama] = m

    # ambang ditetapkan dari sebaran probabilitas pada DATA LATIH
    p_latih = m.predict_proba(cdc[FITUR])[:, 1]
    ambang = np.quantile(p_latih, 1 - cdc.stroke.mean() * 4)

    for label, data in [("CDC (data latih sendiri)", cdc), ("Kaggle (data luar)", kaggle)]:
        p = m.predict_proba(data[FITUR])[:, 1]
        pred = (p >= ambang).astype(int)
        hasil.append({
            "Model": nama, "Diuji pada": label,
            "AUC": roc_auc_score(data.stroke, p),
            "recall": recall_score(data.stroke, pred),
            "precision": precision_score(data.stroke, pred, zero_division=0),
        })

pd.DataFrame(hasil).round(3)

,Model,Diuji pada,AUC,recall,precision
0,Logistic Regression,CDC (data latih sendiri),0.783,0.508,0.127
1,Logistic Regression,Kaggle (data luar),0.799,0.302,0.213
2,Gradient Boosting,CDC (data latih sendiri),0.790,0.522,0.129
3,Gradient Boosting,Kaggle (data luar),0.801,0.297,0.190


### Hasil

Performa **tidak runtuh** ketika model dipindahkan ke sumber data yang sama sekali
berbeda; AUC-nya malah sedikit naik. Ini bukti kuat bahwa model menangkap pola risiko
stroke yang nyata, bukan kekhasan satu dataset.

Kelima artikel acuan kami tidak melakukan validasi eksternal. Inilah yang membedakan
proyek ini.

In [5]:
for nama, m in model_terlatih.items():
    p_latih = m.predict_proba(cdc[FITUR])[:, 1]
    ambang = np.quantile(p_latih, 1 - cdc.stroke.mean() * 4)
    pred = (m.predict_proba(kaggle[FITUR])[:, 1] >= ambang).astype(int)
    print(f"{nama} - confusion matrix pada data luar (Kaggle):")
    print(pd.DataFrame(
        confusion_matrix(kaggle.stroke, pred),
        index=["asli: tidak stroke", "asli: stroke"],
        columns=["prediksi: tidak", "prediksi: stroke"]))
    print()

Logistic Regression - confusion matrix pada data luar (Kaggle):
                    prediksi: tidak  prediksi: stroke
asli: tidak stroke             2963               226
asli: stroke                    141                61



Gradient Boosting - confusion matrix pada data luar (Kaggle):
                    prediksi: tidak  prediksi: stroke
asli: tidak stroke             2933               256
asli: stroke                    142                60



## 4. Kesimpulan dan Batasan

**Kesimpulan.** Validasi silang antar dataset layak dilakukan dan akan menjadi Tugas E
dalam proyek ini.

**Batasan yang wajib ditulis di laporan:**

1. Hanya enam fitur yang tersedia di kedua dataset: **tidak ada kadar glukosa**, dan
   usia hanya berupa kelompok, bukan angka pasti. Karena itu model validasi eksternal
   wajar bila lebih lemah daripada model utama yang memakai 15 fitur.
2. Dataset CDC berbasis **laporan mandiri responden**, bukan rekam medis. Riwayat stroke
   yang dilaporkan bisa saja keliru atau belum terdiagnosis.
3. Kedua dataset berasal dari populasi non-Indonesia. Model tidak boleh diklaim berlaku
   untuk populasi Indonesia tanpa pengujian ulang.

Hal-hal ini dilaporkan apa adanya, bukan disembunyikan.